# KZ Downloader — Colab Runner

Run yt-dlp downloads from any device (phone, Chromebook, someone else's PC) with **no Python install needed**.

**How to use:**
1. On [kzdownloader.pages.dev](https://kzdownloader.pages.dev/), click **☁️ Run in Colab** — this copies your command and opens this notebook.
2. Click the ▶ play button on the cell below.

That's it — no pasting required. The cell reads the command straight from your clipboard, installs yt-dlp/ffmpeg automatically the first time in a session (~20s), then downloads. Every finished file pops up as its own browser download — nothing is saved to Google Drive.

**First time in a browser:** it may ask permission to read your clipboard — click **Allow**. If it can't read the clipboard for any reason, it'll just ask you to paste the command manually instead, so it always works either way.

**Works for:** many public YouTube, Instagram, Facebook, Twitter/X, TikTok, and generic yt-dlp-supported URLs.
**Doesn't work for:** LinkedIn, private/login-gated content, or public videos that YouTube blocks from anonymous Colab sessions — those still need the local Playwright scanner or a local PC run.

In [ ]:
#@title Run (click \u25b6) \u2014 loads your command from clipboard automatically { display-mode: "form" }
#@markdown No need to paste anything \u2014 this reads the command you copied on the KZ Downloader website automatically.
#@markdown If clipboard access is blocked or empty, it'll ask you to paste manually instead.

import os
import shlex
import shutil
import subprocess
import sys
import importlib.util

from google.colab import files as colab_files
from google.colab import output as colab_output

SAVE_DIR = "/content/KZ Downloads"
DENO_BIN_DIR = "/root/.deno/bin"
os.environ["PATH"] = DENO_BIN_DIR + os.pathsep + os.environ.get("PATH", "")
os.makedirs(SAVE_DIR, exist_ok=True)

# \u2500\u2500 Auto-setup: install anything missing (skipped if already present) \u2500\u2500
def ensure_setup():
    need_ytdlp = importlib.util.find_spec("yt_dlp") is None
    need_ffmpeg = shutil.which("ffmpeg") is None
    need_deno = shutil.which("deno") is None
    if need_ytdlp:
        print("\u2699\ufe0f Installing yt-dlp with YouTube JS support (first run only)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp[default]"], check=False)
    if need_ffmpeg:
        print("\u2699\ufe0f Installing ffmpeg (first run only)...")
        subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=False)
    if need_deno:
        print("\u2699\ufe0f Installing Deno for YouTube JS challenges (first run only)...")
        subprocess.run("curl -fsSL https://deno.land/install.sh | sh -s -- -y", shell=True, check=False)
    if need_ytdlp or need_ffmpeg or need_deno:
        print("\u2705 Setup complete.\n")

ensure_setup()

# \u2500\u2500 Grab the command: try the clipboard first, fall back to manual paste \u2500\u2500
def get_clipboard_text():
    try:
        text = colab_output.eval_js('navigator.clipboard.readText()', timeout_sec=10)
        return (text or "").strip()
    except Exception:
        return ""

commands = get_clipboard_text()
looks_like_command = commands.lower().startswith(("yt-dlp", "python -m yt_dlp", "python3 -m yt_dlp"))

if commands and looks_like_command:
    print("\U0001f4cb Using the command copied from KZ Downloader.\n")
else:
    if commands:
        print("\u26a0\ufe0f Clipboard has something, but it doesn't look like a yt-dlp command.")
    else:
        print("\u26a0\ufe0f Couldn't read a command from your clipboard automatically")
        print("   (your browser may need you to allow clipboard access \u2014 try running this cell again).")
    commands = input("\nPaste your command here instead:\n> ").strip()

# \u2500\u2500 Run it \u2500\u2500
def fix_output_path(args, save_dir: str):
    # Strip any existing -o / --output flag and its value (quoted or not)
    cleaned = []
    skip_next = False
    for arg in args:
        if skip_next:
            skip_next = False
            continue
        if arg in ("-o", "--output"):
            skip_next = True
            continue
        if arg.startswith("--output="):
            continue
        cleaned.append(arg)
    template = f'{save_dir}/%(title).150B [%(id)s].%(ext)s'
    return cleaned + ["-o", template]

def display_cmd(args):
    return " ".join(shlex.quote(arg) for arg in args)

def print_failure_hint(output: str):
    text = output.lower()
    if any(needle in text for needle in ("sign in to confirm", "not a bot", "cookies", "confirm you're not a bot")):
        print("\nLikely cause: YouTube is rejecting this anonymous Colab session.")
        print("This is a YouTube/Colab block, not a KZ command problem. Try the same command locally from KZ Downloader on a PC.")
    elif "requested format is not available" in text or "format not available" in text:
        print("\nLikely cause: that format is unavailable for this video. Try changing KZ Downloader's format to Best single file / MP4.")
    elif "ffmpeg" in text and ("not found" in text or "not installed" in text):
        print("\nLikely cause: ffmpeg was unavailable. Re-run this cell so the setup step can install it, then try again.")

def list_files(root):
    found = set()
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            found.add(os.path.join(dirpath, fn))
    return found

lines = [l.strip() for l in commands.splitlines() if l.strip()]

if not lines:
    print("\u26a0\ufe0f No command to run. Copy one from KZ Downloader, then run this cell again.")
else:
    for i, raw_cmd in enumerate(lines, 1):
        cmd = raw_cmd.strip()
        lowered = cmd.lower()
        for prefix in ("python -m yt_dlp", "python3 -m yt_dlp", "yt-dlp"):
            if lowered.startswith(prefix):
                cmd = cmd[len(prefix):].strip()
                break
        try:
            cmd_args = shlex.split(cmd)
        except ValueError as exc:
            print(f"\u26a0\ufe0f Couldn't parse this command: {exc}")
            continue
        cmd_args = fix_output_path(cmd_args, SAVE_DIR)
        full_cmd = [sys.executable, "-m", "yt_dlp", *cmd_args]
        print(f"\n{'='*60}\n[{i}/{len(lines)}] Running:\n{display_cmd(full_cmd)}\n{'='*60}\n")

        before = list_files(SAVE_DIR)
        result = subprocess.run(full_cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if result.stdout:
            print(result.stdout)
        if result.returncode != 0:
            print(f"\u26a0\ufe0f yt-dlp exited with code {result.returncode}. Check the messages above for the exact error.")
            error_lines = [line for line in result.stdout.splitlines() if "error" in line.lower() or "unable" in line.lower()]
            if error_lines:
                print("\nMost relevant yt-dlp message:")
                print("\n".join(error_lines[-5:]))
            print_failure_hint(result.stdout)
        after = list_files(SAVE_DIR)
        new_files = sorted(after - before, key=lambda p: os.path.getmtime(p))
        if not new_files:
            print("\u26a0\ufe0f No new file was created, so there was nothing for Colab to download to your browser.")

        for f in new_files:
            print(f"  \u2b07\ufe0f Downloading: {os.path.basename(f)}")
            colab_files.download(f)

    print("\n\u2705 Done.")

---
### Notes
- **Sharing with non-technical people:** send them the KZ Downloader site link. Clicking **☁️ Run in Colab** there handles opening this notebook for them.
- **Clipboard read requires a secure, recently-focused browser tab.** If the browser blocks it (common on first use, or in some locked-down/managed browsers), the cell falls back to a manual paste prompt automatically — it never gets stuck.
- **Every device / new session** re-runs the install check automatically — it only actually installs anything the first time in that session, and just skips straight to downloading after that.
- **Session limits:** Colab disconnects after ~90 minutes idle and has a session cap of a few hours. Fine for batch downloads, not for anything long-running.
- **Nothing is saved to Google Drive** — files exist only in the temporary Colab session until they're downloaded to your browser, then they're gone from Colab.
- **Private/login-gated pages (LinkedIn, private IG/FB):** these still require the local Playwright scanner (`kz_scanner.py`) run on a PC with a real browser login — Colab has no way to log in interactively.
- **Batch downloads:** if multiple commands were copied (one per line), they'll run one after another, each file downloading individually as it finishes.
- **Colab never auto-runs code on open**, by Google's design — you still click ▶ once. This notebook just removes the paste step so that's the only click needed.